<a href="https://colab.research.google.com/github/crystalloide/IA_102_IA_Agentique/blob/main/IA102_Atelier2_nb2_Superviseur_Comparaison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA102 - Atelier 2 · Notebook 2/2
# Architecture **hiérarchique supervisée** (sous-agents comme outils) + **comparaison** des deux architectures

**Formation IA102 — IA agentique : conception d'applications multi-agents basées sur des LLMs**
Partie 2 — *Patrons d'architectures multi-agents* : « superviser pour mieux déléguer »

---

### 🎯 Objectifs de ce notebook
1. Construire un **superviseur** (agent LangChain `create_agent`) qui orchestre trois **agents spécialisés**
   exposés comme **outils** : `deleguer_recherche`, `deleguer_redaction`, `deleguer_relecture`.
2. Observer l'**orchestration centralisée** : le superviseur planifie, délègue, contrôle et décide de la fin.
3. Mesurer l'architecture sur les **mêmes 3 requêtes**, avec le **même banc de mesure** que le Notebook 1.
4. Produire le **tableau comparatif** Réseau vs Superviseur : trajectoires, appels LLM, tokens, qualité.

### 🧭 Architecture étudiée (2 niveaux hiérarchiques)
```
                              ┌──────────────────────┐
         question ──────────▶ │     SUPERVISEUR      │ ──▶ « TERMINÉ »  (réponse = dernier brouillon validé)
                              │  (create_agent, LLM) │
                              └──────────┬───────────┘
                     outil               │ outil               outil
          ┌──────────────────────────────┼──────────────────────────────┐
          ▼                              ▼                              ▼
 ┌─────────────────┐           ┌─────────────────┐            ┌─────────────────┐
 │ deleguer_       │           │ deleguer_       │            │ deleguer_       │
 │ recherche       │           │ redaction       │            │ relecture       │
 │ → CHERCHEUR     │           │ → RÉDACTEUR     │            │ → RELECTEUR     │
 │ (sous-agent     │           │ (LLM spécialisé)│            │ (LLM spécialisé)│
 │  ReAct + outils │           └────────┬────────┘            └────────┬────────┘
 │  de recherche)  │                    │ écrit                        │ lit
 └────────┬────────┘                    ▼                              ▼
          └── notes ──────────▶  ESPACE DE TRAVAIL PARTAGÉ (notes, brouillon, versions)
```
* **Hub-and-spoke** : les spécialistes ne se parlent jamais ; tout passe par le superviseur.
* **Isolation des contextes** : chaque sous-agent ne reçoit que **sa consigne** (pas l'historique complet).
* **Hiérarchie** : le chercheur est lui-même un agent autonome (boucle ReAct avec ses outils) → 2 niveaux.

### ⏱️ Durée indicative : 1 h 30 — ▶️ Exécution : *Exécution > Tout exécuter* (aucune modification nécessaire)

## 0. Mise en place de l'environnement

### 0.1 Choix du fournisseur LLM et récupération des clés API

Le handlab fonctionne avec **OpenAI**, **Anthropic** ou **Google Gemini** (offre gratuite possible).
Enregistrez votre clé dans les **Secrets** de Colab (icône 🔑 dans la barre de gauche) sous l'un de ces noms,
et activez l'accès pour ce notebook :

| Fournisseur | Nom du secret | Modèles essayés (dans l'ordre) |
|---|---|---|
| OpenAI | `OPENAI_API_KEY` | gpt-4.1-mini → gpt-5-mini → gpt-4o-mini |
| Anthropic | `ANTHROPIC_API_KEY` | claude-haiku-4-5 → claude-sonnet-4-5 |
| Google Gemini | `GOOGLE_API_KEY` (ou `GEMINI_API_KEY`) | gemini-flash-latest → gemini-2.5-flash → gemini-flash-lite-latest |

> 💡 Sans aucune clé, le notebook bascule automatiquement en **mode simulation** (LLM simulé local) :
> tout s'exécute (graphes, handoffs, métriques), mais les textes sont artificiels.
> Pour comparer honnêtement les deux architectures, utilisez **le même fournisseur et le même modèle dans les deux notebooks**.

In [ ]:
# @title 0.1 Clés API et choix du fournisseur
PROVIDER = "auto"  # @param ["auto", "openai", "anthropic", "google_genai", "simulation"]
MODELE = ""        # @param {type:"string"}
# PROVIDER="auto" : premier fournisseur dont une clé est trouvée ; MODELE vide : modèle par défaut.

import os

NOMS_SECRETS = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GEMINI_API_KEY"]
try:
    from google.colab import userdata          # disponible uniquement dans Colab
    for nom in NOMS_SECRETS:
        if not os.environ.get(nom):
            try:
                valeur = userdata.get(nom)
                if valeur:
                    os.environ[nom] = valeur.strip()
            except Exception:
                pass                            # secret absent ou accès non autorisé
except ImportError:
    pass                                        # hors Colab : on utilise les variables d'environnement

if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

CLES = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "google_genai": "GOOGLE_API_KEY"}
FOURNISSEURS_DISPONIBLES = [p for p, v in CLES.items() if os.environ.get(v)]
print("Fournisseurs avec clé détectée :", FOURNISSEURS_DISPONIBLES or "aucun (→ mode simulation)")

### 0.2 Installation des bibliothèques (versions figées)

Les versions sont **figées** (vérifiées ensemble au 21/09/2026) pour éviter toute incompatibilité :
`langgraph 1.2.12`, `langchain 1.4.2`, `langchain-core 1.6.4` et le connecteur du fournisseur détecté.
Seul le connecteur nécessaire est installé, pour limiter les conflits avec les paquets préinstallés de Colab.
Les éventuels messages *« pip's dependency resolver… »* concernant d'autres paquets de Colab, ainsi que
*« you may need to restart the kernel »*, sont sans conséquence : **ne redémarrez pas** l'environnement d'exécution.

> ⚠️ **Quotas** : l'offre gratuite de Gemini peut limiter le nombre de requêtes par jour. Chaque notebook fait
> environ 30 à 70 appels LLM ; en cas d'erreur de quota (429), la requête concernée est marquée en erreur
> et le notebook continue. Une clé payante (OpenAI, Anthropic ou Gemini) garantit une exécution complète.

In [ ]:
# @title 0.2 Installation (≈ 30 s)
PAQUETS = [
    "langgraph==1.2.12", "langgraph-checkpoint==4.2.0", "langgraph-prebuilt==1.1.0", "langgraph-sdk==0.4.5",
    "langchain==1.4.2", "langchain-core==1.6.4", "tabulate", "openpyxl",
]
PAQUETS_FOURNISSEUR = {
    "openai": ["langchain-openai==1.6.3", "openai==3.16.2"],
    "anthropic": ["langchain-anthropic==1.7.2", "anthropic==1.7.0"],
    "google_genai": ["langchain-google-genai==4.4.0", "google-genai==2.24.0"],
}
for p in FOURNISSEURS_DISPONIBLES:
    if PROVIDER in ("auto", p):
        PAQUETS += PAQUETS_FOURNISSEUR[p]
print("Installation de :", " ".join(PAQUETS))
%pip install -q {" ".join(PAQUETS)}

import importlib.metadata as md
for p in ["langgraph", "langchain", "langchain-core", "langchain-openai", "langchain-anthropic", "langchain-google-genai"]:
    try:
        print(f"  {p:24s} {md.version(p)}")
    except md.PackageNotFoundError:
        pass

### 0.3 Génération des fichiers de test

Cette cellule crée la **base documentaire fictive** de l'entreprise *Heliora* (11 documents Markdown :
rapport annuel, note rectificative, études de marché, risques, réglementation, CODIR…) ainsi que le fichier
`requetes_test.json` contenant les **3 requêtes de test** et, pour chacune, les **points clés attendus**
(vérité terrain utilisée pour mesurer la qualité).

Des pièges sont volontairement présents : un chiffre d'affaires **rectifié** (DOC-02 corrige DOC-01),
un document hors sujet (DOC-11), des positions divergentes au CODIR (DOC-09).

In [ ]:
# ============================================================
# Génération des fichiers de test : base documentaire fictive
# de l'entreprise « Heliora » (stockage d'énergie solaire)
# + fichier des 3 requêtes de test avec leurs points clés attendus
# ============================================================
import json, pathlib, textwrap

DOSSIER_CORPUS = pathlib.Path("corpus_heliora")
DOSSIER_CORPUS.mkdir(exist_ok=True)

DOCUMENTS = {
"DOC-01_rapport_annuel_2025.md": """
# Heliora SAS — Rapport annuel 2025 (version publiée le 12 mars 2026)
Identifiant : DOC-01 — Entreprise fictive créée pour l'atelier IA102.

## Chiffres clés 2025
- Chiffre d'affaires consolidé 2025 : 184,6 M€ (2024 : 164,2 M€), soit une croissance de +12,4 %.
- EBITDA : 21,3 M€ (marge de 11,5 %).
- Résultat net : 6,8 M€.
- Effectif au 31/12/2025 : 1 240 salariés.
- Sites de production : Lyon (France), Porto (Portugal), Poznań (Pologne).

## Répartition du chiffre d'affaires par activité
- Onduleurs et électronique de puissance : 41 %.
- Stockage résidentiel (gamme HeliBat) : 34 %.
- Services et maintenance : 25 %.

## Répartition géographique du chiffre d'affaires
France 46 %, Allemagne 21 %, Italie 11 %, Espagne 9 %, autres pays 13 %.

## Faits marquants 2025
- Lancement de la batterie HeliBat X1 (7 kWh) en France et en Italie.
- Signature de 350 contrats de partenariat avec des installateurs en Allemagne (activité onduleurs).
- Annonce de la cession de l'activité « éclairage solaire public » (filiale Heliora Lumen), finalisée en janvier 2026.
""",

"DOC-02_note_rectificative_CA_2025.md": """
# Note rectificative — Chiffre d'affaires 2025 (Direction financière, 20 février 2026)
Identifiant : DOC-02

## Objet
Suite à la cession de la filiale Heliora Lumen (activité « éclairage solaire public »), finalisée en janvier 2026,
cette activité est traitée comme une activité abandonnée (norme IFRS 5). Les comptes 2025 sont retraités.

## Chiffres retraités (activités poursuivies) — CHIFFRES OFFICIELS À UTILISER
- Chiffre d'affaires 2025 retraité : 181,9 M€.
- Chiffre d'affaires 2024 retraité : 164,1 M€.
- Croissance 2025 retraitée : +10,8 %.
- EBITDA 2025 retraité : 21,0 M€ (marge de 11,5 %).

## Consigne
Le chiffre de 184,6 M€ (+12,4 %) publié dans le rapport annuel (DOC-01) inclut Heliora Lumen.
Il ne doit plus être utilisé dans les communications internes ou externes : utiliser 181,9 M€ et +10,8 %.
""",

"DOC-03_fiche_produit_HeliBat_X2.md": """
# Fiche produit — HeliBat X2 (lancement commercial prévu au T2 2027)
Identifiant : DOC-03

## Caractéristiques techniques
- Capacité : 10 kWh par module, extensible jusqu'à 30 kWh (3 modules).
- Chimie : LFP (lithium-fer-phosphate), 6 000 cycles, garantie 12 ans.
- BMS (système de gestion de batterie) de nouvelle génération, développé en interne après l'incident HeliBat X1.

## Économie du produit
- Prix public cible : 6 900 € HT (module 10 kWh).
- Coût de revient industriel : 4 350 € → marge brute de 37 %.
- Capacité de production initiale : 8 000 unités par an (usine de Poznań).

## Certifications nécessaires par pays
- Allemagne : certification réseau VDE-AR-N 4105 — délai de 6 à 8 mois, coût estimé 280 k€.
- Espagne : certification réseau UNE 217001 — délai de 3 mois, coût estimé 90 k€.
- Marquage CE : déjà obtenu.
""",

"DOC-04_etude_marche_Allemagne_2026.md": """
# Étude de marché — Stockage résidentiel en Allemagne (cabinet externe, avril 2026)
Identifiant : DOC-04

## Taille et dynamique du marché
- Segment cible (batteries 8 à 15 kWh) : 2,1 GWh installés en 2025.
- Croissance annuelle prévue 2026-2028 : +9 % par an (marché en voie de maturité).
- Prix moyens de vente en baisse de 8 % par an (forte pression concurrentielle).

## Concurrence
- Plus de 40 fabricants actifs ; les 3 premiers détiennent 55 % du marché.
- Le leader Voltanis (entreprise fictive) vend un système 10 kWh à 6 400 € HT.

## Distribution
- Vente via les installateurs ; marge des distributeurs de 25 à 30 %.
- Heliora dispose déjà d'un réseau de 350 installateurs partenaires (activité onduleurs).

## Potentiel pour Heliora
- Part de marché accessible en 2027 : 1,5 %, soit environ 3 700 unités HeliBat X2.
- Fin de plusieurs aides régionales en 2025 : demande plus sensible au prix.
""",

"DOC-05_etude_marche_Espagne_2026.md": """
# Étude de marché — Stockage résidentiel en Espagne (cabinet externe, avril 2026)
Identifiant : DOC-05

## Taille et dynamique du marché
- Segment cible (batteries 8 à 15 kWh) : 0,9 GWh installés en 2025.
- Croissance annuelle prévue 2026-2028 : +24 % par an (forte dynamique de l'autoconsommation).
- Prix moyens de vente quasi stables : -2 % par an.

## Concurrence
- Environ 15 acteurs, marché peu concentré ; aucun acteur ne dépasse 15 % de part de marché.
- Le concurrent Solenca (entreprise fictive) vend un système 10 kWh à 7 100 € HT.

## Distribution
- Heliora n'a pas de réseau propre en Espagne.
- Partenariat en négociation avec le distributeur Iberdistrib Solar (fictif) : 600 installateurs couverts, commission de 18 %.

## Points d'attention
- Les aides régionales sont versées aux clients avec 9 à 14 mois de délai (risque de report d'achat).

## Potentiel pour Heliora
- Part de marché accessible en 2027 : 4 %, soit environ 3 600 unités HeliBat X2.
""",

"DOC-06_risques_supply_chain.md": """
# Cartographie des risques — Chaîne d'approvisionnement (mise à jour juin 2026)
Identifiant : DOC-06

## Risque n°1 : dépendance à un fournisseur unique de cellules
- 62 % des cellules LFP proviennent d'un seul fournisseur asiatique (contrat jusqu'en mars 2027).
- Impact estimé d'un arrêt d'approvisionnement de 4 semaines : 9 M€ de chiffre d'affaires perdu.
- Atténuation : qualification d'une seconde source européenne (Baltica Cells, fictif) prévue au T3 2026,
  objectif de ramener la dépendance sous 40 % fin 2027 ; stock de sécurité porté de 4 à 6 semaines.

## Risque n°2 : volatilité des prix des matières premières
- Prix des cellules LFP en baisse de 18 % en 2025, mais forte volatilité (lithium).
- Atténuation : contrats à prix indexés sur 12 mois.

## Risque n°3 : logistique maritime
- Délais de fret Asie-Europe passés de 35 à 48 jours en moyenne sur 2025.
""",

"DOC-07_veille_reglementaire_UE.md": """
# Veille réglementaire — Batteries et autoconsommation (juillet 2026)
Identifiant : DOC-07

## Règlement européen sur les batteries (UE) 2023/1542
- Passeport numérique de batterie obligatoire à partir de février 2027 pour les batteries industrielles de plus de 2 kWh,
  catégorie qui inclut les batteries de stockage stationnaire comme la gamme HeliBat.
- Déclaration de l'empreinte carbone et obligations de devoir de vigilance sur la chaîne d'approvisionnement.
- Heliora : projet de mise en conformité lancé, avancement de 40 %, coût estimé à 1,1 M€.
- Risque : sans passeport conforme, impossibilité de mettre sur le marché de nouvelles batteries dans l'UE.

## Espagne
- Cadre de l'autoconsommation favorable (simplification des démarches administratives).

## Allemagne
- Exigences de certification réseau renforcées (norme VDE-AR-N 4105).
""",

"DOC-08_analyse_concurrence.md": """
# Analyse concurrentielle — Stockage résidentiel (mai 2026)
Identifiant : DOC-08

| Concurrent (fictif) | Pays fort | Prix 10 kWh HT | Garantie | Positionnement |
|---|---|---|---|---|
| Voltanis | Allemagne (22 % de part de marché) | 6 400 € | 10 ans | Leader, guerre des prix |
| Brightcell | Import à bas coût | 5 200 € | 8 ans | Entrée de gamme |
| Solenca | Espagne (12 % de part de marché) | 7 100 € | 10 ans | Premium local |
| Heliora HeliBat X2 | — | 6 900 € | 12 ans | Premium, garantie longue |

## Conclusions
- En Allemagne, le prix de l'HeliBat X2 est 8 % au-dessus du leader, dans un marché où les prix baissent de 8 % par an.
- En Espagne, l'HeliBat X2 est moins chère que le premium local Solenca.
- La pression sur les prix est le principal risque commercial en Allemagne.
""",

"DOC-09_compte_rendu_CODIR_juin_2026.md": """
# Compte rendu du comité de direction (CODIR) — 24 juin 2026
Identifiant : DOC-09

## Point 1 : lancement de l'HeliBat X2 en 2027
- Budget de lancement validé : 4,5 M€ (certification, marketing, stocks initiaux).
- Contrainte : un seul pays en phase 1 (2027), le second pays en 2028.
- Critère de décision : point mort commercial atteint en moins de 24 mois.
- Position de la Direction financière : favorable à l'Espagne (certification plus rapide et moins chère, prix stables).
- Position de la Direction commerciale : favorable à l'Allemagne (réseau existant de 350 installateurs, marché plus grand).
- Décision finale attendue au CODIR d'octobre 2026, sur la base d'une note de recommandation.

## Point 2 : suivi de l'incident qualité HeliBat X1
- Voir DOC-10. Le CODIR exige un audit indépendant du nouveau BMS avant tout lancement de l'HeliBat X2.
""",

"DOC-10_incident_qualite_HeliBat_X1.md": """
# Rapport d'incident qualité — HeliBat X1 (mai 2026)
Identifiant : DOC-10

## Description
- Rappel de 1 200 unités HeliBat X1 en France et en Italie suite à un défaut du micrologiciel du BMS
  (mauvaise lecture d'un capteur de température pouvant entraîner une surchauffe). Aucun blessé.
- Coût total estimé du rappel : 2,3 M€.

## Actions correctives
- Correctif du micrologiciel (version 2.4) déployé à distance sur 96 % du parc.
- Revue complète du processus de validation logicielle ; audit indépendant programmé.
- L'HeliBat X2 utilise un BMS de nouvelle génération (voir DOC-03).

## Impacts
- Impact sur l'image de marque en Italie (hausse des annulations de commandes de 6 % en juin 2026).
- Risque de réputation à surveiller pour le lancement de l'HeliBat X2.
""",

"DOC-11_politique_teletravail.md": """
# Politique de télétravail Heliora (document RH, janvier 2026)
Identifiant : DOC-11

- Jusqu'à 2 jours de télétravail par semaine pour les fonctions support.
- Pas de télétravail pour les équipes de production.
- Indemnité forfaitaire de 30 € par mois.
""",
}

for nom, contenu in DOCUMENTS.items():
    (DOSSIER_CORPUS / nom).write_text(textwrap.dedent(contenu).strip() + "\n", encoding="utf-8")

# ------------------------------------------------------------
# Les 3 requêtes de test (complexité croissante).
# « points_cles » = ce qu'une bonne réponse doit contenir ;
# chaque point est validé si AU MOINS UNE des variantes apparaît.
# ------------------------------------------------------------
REQUETES = [
    {
        "id": "R1",
        "type": "Factuelle (1 fait, piège de version)",
        "question": "Quel est le chiffre d'affaires 2025 d'Heliora et sa croissance par rapport à 2024 ?",
        "points_cles": [
            {"libelle": "CA 2025 retraité = 181,9 M€", "variantes": ["181,9", "181.9"]},
            {"libelle": "Croissance retraitée = +10,8 %", "variantes": ["10,8", "10.8"]},
            {"libelle": "Mention du retraitement (cession Heliora Lumen / IFRS 5)", "variantes": ["retrait", "lumen", "ifrs", "rectifi", "cession"]},
        ],
    },
    {
        "id": "R2",
        "type": "Synthèse multi-documents",
        "question": "Rédige une note de synthèse des principaux risques pour Heliora en 2026-2027, avec leur impact et les mesures d'atténuation prévues.",
        "points_cles": [
            {"libelle": "Dépendance fournisseur de cellules (62 %)", "variantes": ["62"]},
            {"libelle": "Incident qualité HeliBat X1 (rappel de 1 200 unités, 2,3 M€)", "variantes": ["1 200", "1200", "2,3 m", "2.3 m", "rappel"]},
            {"libelle": "Passeport batterie UE obligatoire en février 2027", "variantes": ["passeport"]},
            {"libelle": "Pression concurrentielle sur les prix (-8 %/an en Allemagne)", "variantes": ["8 %", "8%", "pression", "prix"]},
            {"libelle": "Mesures d'atténuation (seconde source, stock de sécurité, audit…)", "variantes": ["seconde source", "second source", "stock de securite", "audit", "baltica"]},
        ],
    },
    {
        "id": "R3",
        "type": "Aide à la décision (raisonnement chiffré)",
        "question": "Heliora doit-elle lancer l'HeliBat X2 d'abord en Allemagne ou en Espagne en 2027 ? Donne une recommandation argumentée et chiffrée pour le CODIR.",
        "points_cles": [
            {"libelle": "Taille et croissance des marchés (2,1 GWh / +9 % vs 0,9 GWh / +24 %)", "variantes": ["24 %", "24%", "2,1 gwh", "2.1 gwh", "0,9 gwh", "0.9 gwh"]},
            {"libelle": "Certification (DE : 6-8 mois, 280 k€ / ES : 3 mois, 90 k€)", "variantes": ["280", "90 k", "vde", "une 217001"]},
            {"libelle": "Distribution (350 installateurs DE / Iberdistrib 600 installateurs, 18 %)", "variantes": ["350", "600", "iberdistrib"]},
            {"libelle": "Contraintes CODIR (budget 4,5 M€, point mort < 24 mois)", "variantes": ["4,5", "4.5", "24 mois", "point mort"]},
            {"libelle": "Recommandation explicite", "variantes": ["recommand", "preconis", "nous conseillons", "il est conseille"]},
        ],
    },
]
pathlib.Path("requetes_test.json").write_text(json.dumps(REQUETES, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✅ {len(DOCUMENTS)} documents écrits dans ./{DOSSIER_CORPUS}/")
for f in sorted(DOSSIER_CORPUS.iterdir()):
    print(f"   - {f.name:45s} {f.stat().st_size:6d} octets")
print(f"✅ {len(REQUETES)} requêtes de test écrites dans ./requetes_test.json")

### 0.4 Boîte à outils commune (configuration LLM, outils de recherche, mesures)

Les deux notebooks partagent **exactement** le même code d'infrastructure, écrit dans deux modules :

* `ia102_commun.py` : connexion au LLM (avec repli automatique), **compteur d'appels LLM et de tokens par agent**
  (callbacks LangChain), outils `rechercher_documents` / `lire_document`, évaluation de la qualité
  (couverture des points clés, sources citées, **LLM-juge**) et banc de mesure `mesurer()`.
* `ia102_simulateur.py` : LLM simulé utilisé uniquement en l'absence de clé API (inutile de le lire).

Parcourez `ia102_commun.py` : c'est ce banc de mesure commun qui rend la comparaison équitable.

In [ ]:
%%writefile ia102_commun.py
"""
ia102_commun.py — Boîte à outils commune aux deux notebooks de l'atelier IA102 (partie 2).

Contenu :
  1. configurer()        : choix du fournisseur LLM + test de connexion (repli automatique)
  2. creer_llm(agent)    : fabrique un modèle de chat instrumenté (1 instance par agent)
  3. SUIVI               : compteur d'appels LLM / tokens par agent (callbacks LangChain)
  4. Outils métier       : rechercher_documents, lire_document (sur le corpus local)
  5. Évaluation          : couverture des points clés, sources citées, juge LLM
  6. mesurer()           : exécute une architecture sur une requête et collecte les métriques
"""
from __future__ import annotations

import json, math, os, pathlib, re, time, unicodedata
from collections import Counter, defaultdict
from typing import Any, Callable, Optional

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# ------------------------------------------------------------------
# 1. Configuration du fournisseur LLM
# ------------------------------------------------------------------
# Modèles essayés dans l'ordre (le premier qui répond est retenu).
MODELES_CANDIDATS = {
    "openai": ["gpt-4.1-mini", "gpt-5-mini", "gpt-4o-mini"],
    "anthropic": ["claude-haiku-4-5", "claude-sonnet-4-5"],
    "google_genai": ["gemini-flash-latest", "gemini-2.5-flash", "gemini-flash-lite-latest"],
}
VARIABLE_CLE = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "google_genai": "GOOGLE_API_KEY"}
# Débit max (requêtes / seconde) pour rester sous les quotas (offre gratuite Gemini ≈ 10 req/min).
DEBIT_MAX = {"openai": 2.0, "anthropic": 1.0, "google_genai": 0.15}


class Config:
    provider: str = "simulation"
    modele: str = "simulateur-ia102"
    rate_limiter: Optional[InMemoryRateLimiter] = None

    def __repr__(self):
        return f"Config(provider={self.provider!r}, modele={self.modele!r})"


CONFIG = Config()


def _instancier(provider: str, modele: str, callbacks=None, rate_limiter=None):
    from langchain.chat_models import init_chat_model

    kwargs: dict[str, Any] = {"max_retries": 6}
    if provider == "anthropic":
        kwargs["max_tokens"] = 4096
    if callbacks:
        kwargs["callbacks"] = callbacks
    if rate_limiter is not None:
        kwargs["rate_limiter"] = rate_limiter
    return init_chat_model(modele, model_provider=provider, **kwargs)


def configurer(provider: str = "auto", modele: str = "", verbose: bool = True) -> Config:
    """Choisit le fournisseur et le modèle, vérifie qu'ils répondent, sinon se replie sur le suivant.
    En dernier recours : mode 'simulation' (LLM simulé local, sans clé ni réseau)."""
    provider = (provider or "auto").strip()
    modele = (modele or "").strip()
    if provider == "auto":
        ordre = [p for p in ("openai", "anthropic", "google_genai") if os.environ.get(VARIABLE_CLE[p])]
    elif provider == "simulation":
        ordre = []
    else:
        ordre = [provider]

    for p in ordre:
        if not os.environ.get(VARIABLE_CLE[p]):
            print(f"⚠️  {p} : variable {VARIABLE_CLE[p]} absente, fournisseur ignoré.")
            continue
        candidats = ([modele] if modele else []) + [m for m in MODELES_CANDIDATS[p] if m != modele]
        for m in candidats:
            try:
                reponse = _instancier(p, m).invoke("Réponds uniquement par le mot OK.")
                if verbose:
                    print(f"✅ Connexion réussie : fournisseur={p} | modèle={m} | réponse test={reponse.text[:20]!r}")
                CONFIG.provider, CONFIG.modele = p, m
                CONFIG.rate_limiter = InMemoryRateLimiter(requests_per_second=DEBIT_MAX[p], check_every_n_seconds=0.05, max_bucket_size=1)
                return CONFIG
            except Exception as e:  # modèle retiré, clé invalide, quota…
                print(f"⚠️  {p}/{m} indisponible : {type(e).__name__}: {str(e)[:160]}")

    CONFIG.provider, CONFIG.modele, CONFIG.rate_limiter = "simulation", "simulateur-ia102", None
    print("ℹ️  MODE SIMULATION : aucun LLM réel joignable → un LLM simulé local est utilisé.\n"
          "    Le notebook s'exécute de bout en bout (graphes, handoffs, métriques), mais les textes\n"
          "    produits sont artificiels. Ajoutez une clé API dans les « Secrets » Colab pour un vrai LLM.")
    return CONFIG


# ------------------------------------------------------------------
# 2. Suivi des appels LLM et des tokens
# ------------------------------------------------------------------
class SuiviLLM:
    """Enregistre chaque appel LLM : agent, tokens d'entrée/sortie, durée."""

    def __init__(self):
        self.appels: list[dict] = []
        self.appels_outils: Counter = Counter()

    def reinitialiser(self):
        self.appels.clear()
        self.appels_outils.clear()

    def handler(self, agent: str) -> "GestionnaireSuivi":
        return GestionnaireSuivi(self, agent)

    def resume(self) -> dict:
        par_agent = defaultdict(lambda: {"appels": 0, "tokens": 0})
        for a in self.appels:
            par_agent[a["agent"]]["appels"] += 1
            par_agent[a["agent"]]["tokens"] += a["tokens_entree"] + a["tokens_sortie"]
        te = sum(a["tokens_entree"] for a in self.appels)
        ts = sum(a["tokens_sortie"] for a in self.appels)
        return {
            "nb_appels_llm": len(self.appels),
            "tokens_entree": te,
            "tokens_sortie": ts,
            "tokens_total": te + ts,
            "appels_par_agent": {k: v["appels"] for k, v in par_agent.items()},
            "tokens_par_agent": {k: v["tokens"] for k, v in par_agent.items()},
            "nb_appels_outils": sum(self.appels_outils.values()),
        }


class GestionnaireSuivi(BaseCallbackHandler):
    def __init__(self, suivi: SuiviLLM, agent: str):
        self.suivi, self.agent, self._t0 = suivi, agent, {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self._t0[run_id] = time.perf_counter()

    def on_llm_end(self, response, *, run_id, **kwargs):
        te = ts = 0
        try:
            msg = response.generations[0][0].message
            usage = getattr(msg, "usage_metadata", None) or {}
            te, ts = int(usage.get("input_tokens", 0) or 0), int(usage.get("output_tokens", 0) or 0)
        except Exception:
            pass
        if te == 0 and ts == 0:  # repli : certains fournisseurs mettent l'usage dans llm_output
            u = (response.llm_output or {}).get("token_usage") or (response.llm_output or {}).get("usage") or {}
            te = int(u.get("prompt_tokens", u.get("input_tokens", 0)) or 0)
            ts = int(u.get("completion_tokens", u.get("output_tokens", 0)) or 0)
        self.suivi.appels.append({
            "agent": self.agent, "tokens_entree": te, "tokens_sortie": ts,
            "duree_s": round(time.perf_counter() - self._t0.pop(run_id, time.perf_counter()), 2),
        })


SUIVI = SuiviLLM()          # suivi des architectures comparées
SUIVI_JUGE = SuiviLLM()     # suivi séparé pour le juge (exclu des métriques)


def creer_llm(agent: str, suivi: SuiviLLM = None):
    """Crée un modèle de chat dédié à un agent : ses appels sont comptés sous le nom `agent`."""
    suivi = suivi or SUIVI
    callbacks = [suivi.handler(agent)]
    if CONFIG.provider == "simulation":
        from ia102_simulateur import LLMSimule
        return LLMSimule(agent=agent, callbacks=callbacks)
    return _instancier(CONFIG.provider, CONFIG.modele, callbacks=callbacks, rate_limiter=CONFIG.rate_limiter)


def texte(msg: Any) -> str:
    """Texte d'un message, quel que soit le format du contenu (chaîne ou liste de blocs)."""
    if isinstance(msg, BaseMessage):
        return msg.text or ""
    return str(msg)


# ------------------------------------------------------------------
# 3. Outils métier : recherche dans le corpus local
# ------------------------------------------------------------------
DOSSIER_CORPUS = pathlib.Path("corpus_heliora")
MOTS_VIDES = set("""a au aux avec ce ces dans de des du elle en et est il ils je la le les leur lui ma mais me
mes mon ne nos notre nous on ou par pas pour qu que qui sa se ses son sur ta te tes ton tu un une vos votre vous
y d l s n c j t m qu donne doit etre faire quel quelle quels quelles quoi comment rediges redige avec sont plus
d'abord abord""".split())


def normaliser(t: str) -> str:
    t = unicodedata.normalize("NFKD", t.replace(" ", " ").replace("\xa0", " "))
    t = "".join(c for c in t if not unicodedata.combining(c)).lower()
    t = re.sub(r"(?<=\d),(?=\d)", ".", t)                 # 181,9 -> 181.9
    t = re.sub(r"(?<=\b\d)\s(?=\d{3}\b)|(?<=\b\d\d)\s(?=\d{3}\b)|(?<=\b\d\d\d)\s(?=\d{3}\b)", "", t)  # 1 200 -> 1200
    return t


def _mots(t: str) -> list[str]:
    return [m for m in re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", normaliser(t)) if m not in MOTS_VIDES and len(m) > 1]


class Corpus:
    def __init__(self, dossier: pathlib.Path = DOSSIER_CORPUS):
        self.docs: dict[str, dict] = {}
        self.passages: list[dict] = []
        for f in sorted(pathlib.Path(dossier).glob("*.md")):
            doc_id = f.name[:6]
            contenu = f.read_text(encoding="utf-8")
            titre = contenu.splitlines()[0].lstrip("# ").strip()
            self.docs[doc_id] = {"titre": titre, "contenu": contenu, "fichier": f.name}
            for bloc in re.split(r"\n(?=## )", contenu):
                self.passages.append({"doc_id": doc_id, "titre": titre, "texte": bloc.strip(), "mots": _mots(titre + " " + bloc)})
        n = len(self.passages) or 1
        df = Counter(m for p in self.passages for m in set(p["mots"]))
        self.idf = {m: math.log(1 + n / c) for m, c in df.items()}

    def rechercher(self, requete: str, k: int = 4) -> list[dict]:
        q = set(_mots(requete))
        scores = []
        for p in self.passages:
            tf = Counter(p["mots"])
            s = sum((1 + math.log(tf[m])) * self.idf.get(m, 0) for m in q if tf[m])
            if s > 0:
                scores.append((s / math.sqrt(len(p["mots"]) + 1), p))
        scores.sort(key=lambda x: -x[0])
        return [p for _, p in scores[:k]]


_CORPUS: Optional[Corpus] = None


def corpus() -> Corpus:
    global _CORPUS
    if _CORPUS is None:
        _CORPUS = Corpus()
    return _CORPUS


@tool
def rechercher_documents(requete: str) -> str:
    """Recherche dans la base documentaire interne d'Heliora (rapports, études de marché, risques,
    réglementation, CODIR…). Renvoie les 4 passages les plus pertinents avec leur identifiant [DOC-xx].
    Utilise des mots-clés précis (ex. : 'chiffre d'affaires 2025 retraité')."""
    SUIVI.appels_outils["rechercher_documents"] += 1
    resultats = corpus().rechercher(requete)
    if not resultats:
        return "Aucun passage trouvé. Reformule avec d'autres mots-clés."
    return "\n\n".join(f"[{p['doc_id']}] {p['titre']}\n{p['texte']}" for p in resultats)


@tool
def lire_document(doc_id: str) -> str:
    """Lit le contenu intégral d'un document à partir de son identifiant (ex. : 'DOC-03')."""
    SUIVI.appels_outils["lire_document"] += 1
    doc_id = doc_id.strip().strip("[]").upper()
    d = corpus().docs.get(doc_id)
    if d is None:
        return f"Document {doc_id} introuvable. Identifiants valides : {', '.join(corpus().docs)}"
    return f"[{doc_id}] {d['contenu']}"


OUTILS_RECHERCHE = [rechercher_documents, lire_document]


# ------------------------------------------------------------------
# 4. Évaluation de la qualité
# ------------------------------------------------------------------
def charger_requetes(chemin: str = "requetes_test.json") -> list[dict]:
    return json.loads(pathlib.Path(chemin).read_text(encoding="utf-8"))


def couverture_points_cles(reponse: str, requete: dict) -> tuple[float, list[str]]:
    """Proportion des points clés attendus présents dans la réponse (contrôle automatique par mots-clés)."""
    r = normaliser(reponse)
    manquants = [pc["libelle"] for pc in requete["points_cles"]
                 if not any(normaliser(v) in r for v in pc["variantes"])]
    n = len(requete["points_cles"])
    return round((n - len(manquants)) / n, 2), manquants


def sources_citees(reponse: str) -> list[str]:
    valides = set(corpus().docs)
    return sorted({d for d in re.findall(r"DOC-\d\d", reponse.upper()) if d in valides})


class EvaluationJuge(BaseModel):
    """Évaluation d'une réponse par un LLM-juge (notes de 1 à 5)."""
    exactitude: int = Field(description="Exactitude des faits et chiffres par rapport aux points clés (1 à 5)")
    completude: int = Field(description="Couverture des points clés attendus (1 à 5)")
    clarte: int = Field(description="Clarté, structure et concision (1 à 5)")
    sourcage: int = Field(description="Qualité des citations de sources [DOC-xx] (1 à 5)")
    commentaire: str = Field(description="Justification en une ou deux phrases")


PROMPT_JUGE = """Tu es un évaluateur exigeant et impartial. Évalue la RÉPONSE à la QUESTION
en t'appuyant sur les POINTS CLÉS ATTENDUS (vérité terrain). Pénalise les chiffres faux ou périmés
(ex. : utiliser un chiffre non retraité alors qu'un chiffre retraité existe).

QUESTION : {question}

POINTS CLÉS ATTENDUS :
{points}

RÉPONSE À ÉVALUER :
{reponse}

Donne des notes entières de 1 (très mauvais) à 5 (excellent)."""


def juger(reponse: str, requete: dict) -> dict:
    """Note de qualité par LLM-juge (mêmes critères pour les deux architectures).
    En mode simulation : note heuristique dérivée de la couverture et des sources."""
    cov, _ = couverture_points_cles(reponse, requete)
    if CONFIG.provider == "simulation":
        n = 1 + round(4 * cov)
        s = min(5, 1 + 2 * len(sources_citees(reponse)))
        return {"exactitude": n, "completude": n, "clarte": 3, "sourcage": s,
                "commentaire": "Note heuristique (mode simulation, pas de LLM-juge).", "note_juge": round((n + n + 3 + s) / 4, 2)}
    points = "\n".join(f"- {pc['libelle']}" for pc in requete["points_cles"])
    juge = creer_llm("juge", suivi=SUIVI_JUGE).with_structured_output(EvaluationJuge)
    for tentative in range(3):
        try:
            ev = juge.invoke(PROMPT_JUGE.format(question=requete["question"], points=points, reponse=reponse[:12000]))
            d = ev.model_dump() if hasattr(ev, "model_dump") else dict(ev)
            for k in ("exactitude", "completude", "clarte", "sourcage"):
                d[k] = max(1, min(5, int(d[k])))
            d["note_juge"] = round((d["exactitude"] + d["completude"] + d["clarte"] + d["sourcage"]) / 4, 2)
            return d
        except Exception as e:
            err = e
            time.sleep(2)
    return {"exactitude": None, "completude": None, "clarte": None, "sourcage": None,
            "commentaire": f"Échec du juge : {type(err).__name__}", "note_juge": None}


# ------------------------------------------------------------------
# 5. Banc de mesure commun
# ------------------------------------------------------------------
def mesurer(architecture: str, executer: Callable[[str], dict], requete: dict, afficher: bool = True) -> dict:
    """Exécute `executer(question)` et renvoie un dictionnaire de métriques homogène.
    `executer` doit renvoyer au minimum {'reponse': str, 'trajectoire': list[str]}."""
    SUIVI.reinitialiser()
    t0 = time.perf_counter()
    try:
        sortie = executer(requete["question"])
        erreur = None
    except Exception as e:
        sortie = {"reponse": "", "trajectoire": ["ERREUR"]}
        erreur = f"{type(e).__name__}: {e}"
        print(f"❌ Erreur pendant l'exécution : {erreur}")
    duree = round(time.perf_counter() - t0, 1)
    stats = SUIVI.resume()
    reponse = sortie.get("reponse") or ""
    cov, manquants = couverture_points_cles(reponse, requete)
    resultat = {
        "architecture": architecture,
        "requete_id": requete["id"],
        "type_requete": requete["type"],
        "question": requete["question"],
        "provider": CONFIG.provider,
        "modele": CONFIG.modele,
        "trajectoire": sortie.get("trajectoire", []),
        "nb_etapes": sum(1 for e in sortie.get("trajectoire", []) if e != "FIN" and not e.startswith(("ARRÊT", "ERREUR"))),
        **stats,
        "duree_s": duree,
        "reponse": reponse,
        "nb_mots_reponse": len(reponse.split()),
        "couverture_points_cles": cov,
        "points_manquants": manquants,
        "sources_citees": sources_citees(reponse),
        "erreur": erreur,
    }
    resultat.update({k: v for k, v in sortie.items() if k not in resultat and k != "reponse"})
    resultat["evaluation_juge"] = juger(reponse, requete) if reponse else {"note_juge": None, "commentaire": "Réponse vide"}
    resultat["note_juge"] = resultat["evaluation_juge"].get("note_juge")
    if afficher:
        afficher_resultat(resultat)
    return resultat


def afficher_resultat(r: dict):
    print("=" * 100)
    print(f"[{r['architecture']}] {r['requete_id']} — {r['question']}")
    print("-" * 100)
    print("Trajectoire      :", " → ".join(r["trajectoire"]))
    print(f"Appels LLM       : {r['nb_appels_llm']}  (par agent : {r['appels_par_agent']})")
    print(f"Tokens           : {r['tokens_total']} (entrée {r['tokens_entree']} / sortie {r['tokens_sortie']})")
    print(f"Appels d'outils  : {r['nb_appels_outils']}   |   Durée : {r['duree_s']} s")
    print(f"Qualité          : couverture points clés = {r['couverture_points_cles']:.0%} | "
          f"sources citées = {r['sources_citees']} | note juge = {r['note_juge']}")
    if r["points_manquants"]:
        print("Points manquants :", "; ".join(r["points_manquants"]))


def sauvegarder_resultats(resultats: list[dict], chemin: str):
    pathlib.Path(chemin).write_text(json.dumps(resultats, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"💾 {len(resultats)} résultats sauvegardés dans {chemin}")


def tableau_metriques(resultats: list[dict]):
    import pandas as pd
    lignes = []
    for r in resultats:
        lignes.append({
            "Architecture": r["architecture"], "Requête": r["requete_id"], "Type": r["type_requete"],
            "Trajectoire": " → ".join(r["trajectoire"]), "Étapes": r["nb_etapes"],
            "Appels LLM": r["nb_appels_llm"], "Tokens entrée": r["tokens_entree"],
            "Tokens sortie": r["tokens_sortie"], "Tokens total": r["tokens_total"],
            "Appels outils": r["nb_appels_outils"], "Durée (s)": r["duree_s"],
            "Couverture points clés": r["couverture_points_cles"], "Nb sources citées": len(r["sources_citees"]),
            "Note juge (/5)": r["note_juge"], "Mots réponse": r["nb_mots_reponse"],
        })
    return pd.DataFrame(lignes)

In [ ]:
%%writefile ia102_simulateur.py
"""
ia102_simulateur.py — LLM SIMULÉ (déterministe, hors-ligne) utilisé uniquement quand aucune clé API
n'est disponible. Il imite le comportement attendu de chaque agent (appels d'outils, transferts,
délégations, rédaction à partir des notes) pour que les graphes s'exécutent de bout en bout.
⚠️ Les textes produits sont artificiels : les métriques de qualité n'ont de sens qu'avec un vrai LLM.
Vous n'avez PAS besoin de lire ce fichier pour l'atelier.
"""
from __future__ import annotations

import re, uuid
from typing import Any, Optional

from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool


def _txt(m: BaseMessage) -> str:
    return m.text if isinstance(m, BaseMessage) else str(m)


def _appel(nom: str, **args) -> dict:
    return {"name": nom, "args": args, "id": "call_" + uuid.uuid4().hex[:12], "type": "tool_call"}


def _section(t: str, label: str) -> str:
    m = re.search(re.escape(label) + r"\s*:?\s*\n?(.*?)(?=\n[A-ZÉÈÀ' ]{4,}\s*:|\Z)", t, re.S)
    return m.group(1).strip() if m else ""


def _faits(notes: str, question: str, n: int = 8) -> list[tuple[str, str]]:
    """Extrait des notes les lignes factuelles (avec chiffres) les plus liées à la question."""
    from ia102_commun import _mots
    q = set(_mots(question))
    faits, vus = [], set()
    doc = "DOC-??"
    for ligne in notes.splitlines():
        m = re.search(r"\[?(DOC-\d\d)\]?", ligne)
        if m and ligne.lstrip().startswith(("[", "#")):
            doc = m.group(1)
        l = ligne.strip().lstrip("-*| ").strip()
        if len(l) < 25 or not re.search(r"\d", l) or l.startswith(("#", "[DOC", "Identifiant")):
            continue
        cle = l[:60]
        if cle in vus:
            continue
        vus.add(cle)
        score = len(q & set(_mots(l)))
        faits.append((score, l, doc))
    faits.sort(key=lambda x: -x[0])
    return [(l, d) for _, l, d in faits[:n]]


def rediger(question: str, notes: str, avec_sources: bool) -> str:
    faits = _faits(notes, question)
    lignes = [f"**Réponse (texte généré par le LLM SIMULÉ)** — {question}", ""]
    for l, d in faits:
        lignes.append(f"- {l}" + (f" [{d}]" if avec_sources else ""))
    if "recommand" in question.lower():
        lignes += ["", "Recommandation : lancer d'abord en Espagne (certification plus rapide et moins chère, "
                       "croissance de +24 %/an), puis l'Allemagne en 2028."]
    if avec_sources:
        docs = sorted({d for _, d in faits if d != "DOC-??"})
        lignes += ["", "Sources : " + ", ".join(docs)]
    return "\n".join(lignes)


class LLMSimule(BaseChatModel):
    agent: str = "agent"

    @property
    def _llm_type(self) -> str:
        return "llm-simule-ia102"

    def bind_tools(self, tools, *, tool_choice: Optional[str] = None, **kwargs):
        schemas = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=schemas, tool_choice=tool_choice, **kwargs)

    # --------------------------------------------------------------
    def _generate(self, messages: list[BaseMessage], stop=None, run_manager=None, tools=None, **kwargs) -> ChatResult:
        noms = [t["function"]["name"] for t in (tools or [])]
        if any(n.startswith("transferer_vers_") or n == "publier_reponse_finale" for n in noms):
            msg = self._agent_reseau(messages)
        elif any(n.startswith("deleguer_") for n in noms):
            msg = self._superviseur(messages)
        elif "rechercher_documents" in noms:
            msg = self._chercheur_sous_agent(messages)
        else:
            msg = self._sans_outil(messages)
        entree = sum(len(_txt(m)) + len(str(getattr(m, "tool_calls", ""))) for m in messages) + len(str(tools or ""))
        sortie = len(_txt(msg)) + len(str(msg.tool_calls))
        msg.usage_metadata = {"input_tokens": entree // 4, "output_tokens": sortie // 4, "total_tokens": (entree + sortie) // 4}
        return ChatResult(generations=[ChatGeneration(message=msg)])

    # --------------------------------------------------------------
    @staticmethod
    def _question(messages) -> str:
        for m in messages:
            if isinstance(m, HumanMessage):
                return _txt(m)
        return ""

    @staticmethod
    def _args_dernier_appel(messages, nom_outil: str) -> Optional[dict]:
        for m in reversed(messages):
            for tc in getattr(m, "tool_calls", None) or []:
                if tc["name"] == nom_outil:
                    return tc["args"]
        return None

    def _tour_courant(self, messages) -> list[BaseMessage]:
        """Messages produits depuis que l'agent a reçu la main."""
        debut = 0
        for i, m in enumerate(messages):
            if isinstance(m, ToolMessage) and m.name == f"transferer_vers_{self.agent}":
                debut = i + 1
        return messages[debut:]

    def _agent_reseau(self, messages) -> AIMessage:
        q = self._question(messages)
        tour = self._tour_courant(messages)
        if self.agent == "chercheur":
            recherches = [m for m in tour if isinstance(m, ToolMessage) and m.name == "rechercher_documents"]
            if len(recherches) == 0:
                consigne = (self._args_dernier_appel(messages, "transferer_vers_chercheur") or {}).get("message", "")
                return AIMessage(content="", tool_calls=[_appel("rechercher_documents", requete=f"{q} {consigne}".strip())])
            if len(recherches) == 1 and len(q) > 100:
                return AIMessage(content="", tool_calls=[_appel("rechercher_documents", requete="chiffres clés impacts " + q[-80:])])
            notes = "\n\n".join(_txt(m) for m in recherches)[:3500]
            return AIMessage(content="", tool_calls=[_appel("transferer_vers_redacteur", message="Notes de recherche :\n" + notes)])
        if self.agent == "redacteur":
            demande = (self._args_dernier_appel(messages, "transferer_vers_redacteur") or {}).get("message", "")
            notes = "\n".join(_txt(m) for m in messages if isinstance(m, ToolMessage) and m.name == "rechercher_documents") + "\n" + demande
            revision = self._args_dernier_appel(messages, "transferer_vers_relecteur") is not None
            brouillon = rediger(q, notes, avec_sources=revision or len(q) < 100)
            return AIMessage(content="", tool_calls=[_appel("transferer_vers_relecteur", message=brouillon)])
        # relecteur
        brouillon = (self._args_dernier_appel(messages, "transferer_vers_relecteur") or {}).get("message", "")
        if "Sources :" not in brouillon:
            return AIMessage(content="", tool_calls=[_appel("transferer_vers_redacteur",
                message="À RÉVISER : cite chaque fait avec sa source [DOC-xx] et ajoute une liste des sources.")])
        return AIMessage(content="", tool_calls=[_appel("publier_reponse_finale", commentaire="Brouillon validé.")])

    def _superviseur(self, messages) -> AIMessage:
        q = self._question(messages)
        outils = [m for m in messages if isinstance(m, ToolMessage)]
        n = {k: sum(1 for m in outils if m.name == k) for k in ("deleguer_recherche", "deleguer_redaction", "deleguer_relecture")}
        dernier = outils[-1] if outils else None
        if n["deleguer_recherche"] == 0:
            return AIMessage(content="", tool_calls=[_appel("deleguer_recherche", consigne=f"Trouve les faits et chiffres utiles pour : {q}")])
        if n["deleguer_redaction"] == 0:
            return AIMessage(content="", tool_calls=[_appel("deleguer_redaction", consignes="Rédige une réponse structurée et chiffrée.")])
        if dernier.name == "deleguer_redaction":
            return AIMessage(content="", tool_calls=[_appel("deleguer_relecture")])
        if dernier.name == "deleguer_relecture" and _txt(dernier).startswith("À RÉVISER") and n["deleguer_redaction"] < 3:
            return AIMessage(content="", tool_calls=[_appel("deleguer_redaction", consignes=_txt(dernier))])
        return AIMessage(content="TERMINÉ — la réponse finale a été validée par le relecteur.")

    def _chercheur_sous_agent(self, messages) -> AIMessage:
        q = self._question(messages)
        recherches = [m for m in messages if isinstance(m, ToolMessage)]
        if len(recherches) == 0:
            return AIMessage(content="", tool_calls=[_appel("rechercher_documents", requete=q)])
        if len(recherches) == 1 and len(q) > 140:
            return AIMessage(content="", tool_calls=[_appel("rechercher_documents", requete="chiffres clés impacts " + q[-80:])])
        return AIMessage(content="Notes de recherche :\n" + "\n\n".join(_txt(m) for m in recherches)[:3500])

    def _sans_outil(self, messages) -> AIMessage:
        t = "\n".join(_txt(m) for m in messages if not isinstance(m, SystemMessage))
        q = _section(t, "QUESTION")
        if self.agent == "relecteur":
            brouillon = _section(t, "BROUILLON À RELIRE")
            if "Sources :" not in brouillon:
                return AIMessage(content="À RÉVISER : cite chaque fait avec sa source [DOC-xx] et ajoute une liste des sources.")
            return AIMessage(content="VALIDÉ : le brouillon est exact, sourcé et répond à la question.")
        if self.agent == "redacteur":
            precedent = _section(t, "BROUILLON PRÉCÉDENT")
            revision = bool(precedent) and not precedent.startswith("(aucun")
            return AIMessage(content=rediger(q, _section(t, "NOTES DE RECHERCHE"), avec_sources=revision or len(q) < 100))
        return AIMessage(content="OK")

In [ ]:
# @title 0.5 Connexion au LLM et test des outils
import importlib, sys
sys.path.insert(0, os.getcwd())
import ia102_commun, ia102_simulateur
importlib.reload(ia102_commun); importlib.reload(ia102_simulateur)
from ia102_commun import (CONFIG, SUIVI, configurer, creer_llm, texte, rechercher_documents, lire_document,
                          OUTILS_RECHERCHE, charger_requetes, mesurer, afficher_resultat,
                          sauvegarder_resultats, tableau_metriques)

configurer(PROVIDER, MODELE)
REQUETES = charger_requetes()

print("\n--- Test de l'outil de recherche ---")
print(rechercher_documents.invoke({"requete": "chiffre d'affaires 2025 retraité"})[:700], "…")

## 1. L'espace de travail partagé et les prompts des spécialistes

Pour éviter que le superviseur ne recopie de longs textes d'un agent à l'autre (effet « téléphone arabe » et
surcoût en tokens), les livrables (notes, brouillon) sont déposés dans un **espace de travail partagé** ;
le superviseur ne reçoit que des **comptes rendus courts**.

Chaque spécialiste a un prompt **focalisé sur sa tâche** — il ignore l'existence des autres agents.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.errors import GraphRecursionError

ESPACE: dict = {}
MAX_DELEGATIONS = 10   # garde-fou : nombre max de délégations du superviseur par requête


def reinitialiser_espace(question: str):
    ESPACE.clear()
    ESPACE.update(question=question, notes=[], brouillon="", versions=0, verdicts=[], delegations=0)


def limite_atteinte() -> str:
    """Garde-fou : compte les délégations et renvoie un message d'arrêt au-delà de MAX_DELEGATIONS."""
    ESPACE["delegations"] += 1
    if ESPACE["delegations"] > MAX_DELEGATIONS:
        return "LIMITE DE DÉLÉGATIONS ATTEINTE : ne délègue plus, réponds maintenant par « TERMINÉ »."
    return ""


PROMPT_CHERCHEUR_SUP = """Tu es un CHERCHEUR documentaire pour l'entreprise Heliora.
Utilise rechercher_documents (plusieurs requêtes courtes et ciblées si besoin) et lire_document pour collecter
les faits et chiffres demandés dans la consigne. Attention aux versions : si un document rectifie un chiffre
d'un autre, retiens le chiffre rectifié et signale-le. Ignore les documents hors sujet.
Réponds par des notes structurées et concises : un fait ou chiffre par puce, chacun suivi de sa source [DOC-xx]."""

PROMPT_REDACTEUR_SUP = """Tu es un RÉDACTEUR professionnel. Rédige en français la réponse à la QUESTION à partir des
NOTES DE RECHERCHE uniquement (n'invente rien), en appliquant les CONSIGNES et en améliorant le BROUILLON PRÉCÉDENT s'il existe.
Réponse structurée (titres courts, puces), précise et chiffrée ; chaque fait est suivi de sa source [DOC-xx] ;
termine par une ligne « Sources : … ». Renvoie uniquement le texte de la réponse."""

PROMPT_RELECTEUR_SUP = """Tu es un RELECTEUR exigeant (contrôle qualité). Vérifie le BROUILLON À RELIRE : exactitude des
chiffres par rapport aux NOTES DE RECHERCHE (chiffres rectifiés / à jour), réponse complète à la QUESTION, sources [DOC-xx]
citées, clarté. Ta réponse commence OBLIGATOIREMENT par :
- « VALIDÉ : » suivi d'une phrase de justification, si le brouillon est satisfaisant ;
- « À RÉVISER : » suivi de la liste précise des corrections (et des informations manquantes à rechercher), sinon.
Sois exigeant mais pragmatique."""

GABARIT_REDACTION = """QUESTION :
{question}

NOTES DE RECHERCHE :
{notes}

BROUILLON PRÉCÉDENT :
{precedent}

CONSIGNES :
{consignes}"""

GABARIT_RELECTURE = """QUESTION :
{question}

NOTES DE RECHERCHE :
{notes}

BROUILLON À RELIRE :
{brouillon}

POINTS D'ATTENTION :
{points}"""

## 2. Les agents spécialisés… exposés comme des **outils**

C'est le cœur du patron « **sous-agents comme outils** » : pour le superviseur, un agent n'est qu'un outil
avec un nom, une description et des paramètres. L'outil encapsule l'appel complet au sous-agent
(qui peut lui-même faire plusieurs appels LLM et utiliser ses propres outils) et ne renvoie qu'un **résultat condensé**.

* `deleguer_recherche(consigne)` → **sous-agent chercheur** créé avec `create_agent` (boucle ReAct + outils de recherche) ;
* `deleguer_redaction(consignes)` → **rédacteur** (1 appel LLM) ; le brouillon est déposé dans l'espace partagé ;
* `deleguer_relecture(points_d_attention)` → **relecteur** (1 appel LLM) ; renvoie « VALIDÉ » ou « À RÉVISER ».

In [ ]:
agent_chercheur = create_agent(creer_llm("chercheur"), tools=OUTILS_RECHERCHE,
                               system_prompt=PROMPT_CHERCHEUR_SUP, name="chercheur")
llm_redacteur = creer_llm("redacteur")
llm_relecteur = creer_llm("relecteur")


@tool
def deleguer_recherche(consigne: str) -> str:
    """Délègue une mission de recherche documentaire au CHERCHEUR (agent autonome qui interroge la base
    documentaire d'Heliora). `consigne` : ce qu'il faut trouver, de façon précise.
    Renvoie ses notes sourcées (elles sont aussi transmises automatiquement au rédacteur)."""
    if stop := limite_atteinte():
        return stop
    etat = {"messages": []}
    try:
        for etat in agent_chercheur.stream({"messages": [HumanMessage(consigne)]}, {"recursion_limit": 25},
                                           stream_mode="values"):
            pass
        notes = texte(etat["messages"][-1]).strip()
    except GraphRecursionError:   # le chercheur a trop bouclé : on garde les résultats bruts obtenus
        notes = "\n\n".join(texte(m) for m in etat["messages"] if isinstance(m, ToolMessage))[:6000]
    notes = notes or "(aucune note)"
    ESPACE["notes"].append(notes)
    return notes


@tool
def deleguer_redaction(consignes: str) -> str:
    """Délègue au RÉDACTEUR la rédaction (ou la révision) de la réponse finale. Il reçoit automatiquement la question,
    toutes les notes de recherche et le brouillon précédent. `consignes` : format attendu, angle, ou corrections
    demandées par le relecteur. Renvoie un court compte rendu ; le brouillon est enregistré dans l'espace partagé."""
    if stop := limite_atteinte():
        return stop
    prompt = GABARIT_REDACTION.format(question=ESPACE["question"],
                                      notes="\n\n".join(ESPACE["notes"]) or "(aucune)",
                                      precedent=ESPACE["brouillon"] or "(aucun)",
                                      consignes=consignes or "(aucune)")
    brouillon = texte(llm_redacteur.invoke([SystemMessage(PROMPT_REDACTEUR_SUP), HumanMessage(prompt)])).strip()
    ESPACE["brouillon"] = brouillon
    ESPACE["versions"] += 1
    return (f"Brouillon v{ESPACE['versions']} enregistré ({len(brouillon.split())} mots). "
            f"Début : {brouillon[:250]}…")


@tool
def deleguer_relecture(points_d_attention: str = "") -> str:
    """Demande au RELECTEUR de contrôler le dernier brouillon enregistré (exactitude, complétude, sources, clarté).
    `points_d_attention` : éléments à vérifier en priorité (facultatif).
    Renvoie un verdict commençant par « VALIDÉ » ou « À RÉVISER » avec la liste des corrections."""
    if stop := limite_atteinte():
        return stop
    if not ESPACE["brouillon"]:
        return "À RÉVISER : aucun brouillon n'a encore été rédigé."
    prompt = GABARIT_RELECTURE.format(question=ESPACE["question"],
                                      notes="\n\n".join(ESPACE["notes"]) or "(aucune)",
                                      brouillon=ESPACE["brouillon"], points=points_d_attention or "(aucun)")
    verdict = texte(llm_relecteur.invoke([SystemMessage(PROMPT_RELECTEUR_SUP), HumanMessage(prompt)])).strip()
    ESPACE["verdicts"].append(verdict)
    return verdict


OUTILS_SOUS_AGENTS = [deleguer_recherche, deleguer_redaction, deleguer_relecture]

## 3. Le superviseur

Le superviseur est un agent `create_agent` dont les **seuls outils sont les sous-agents**. Il ne cherche pas et n'écrit pas :
il **planifie**, **délègue**, **contrôle** le verdict du relecteur et décide **quand s'arrêter**.

In [ ]:
PROMPT_SUPERVISEUR = """Tu es le SUPERVISEUR d'une équipe de trois agents spécialisés qui répond aux questions de la direction
de l'entreprise Heliora à partir de sa base documentaire interne.
Tu ne fais aucune recherche et tu n'écris jamais la réponse toi-même : tu DÉLÈGUES via tes outils, une seule délégation à la fois.

Processus :
1. deleguer_recherche avec une consigne précise (tu peux relancer une recherche ciblée si des éléments manquent) ;
2. deleguer_redaction avec des consignes de format adaptées à la question ;
3. deleguer_relecture ;
4. si le verdict est « À RÉVISER » : deleguer_redaction avec les corrections (ou deleguer_recherche si des faits manquent),
   puis nouvelle relecture — au maximum 2 cycles de révision.
Quand le relecteur a VALIDÉ le brouillon (ou après 2 cycles de révision), réponds par un court message commençant par
« TERMINÉ » : la réponse finale est le dernier brouillon enregistré, ne le recopie pas."""

superviseur = create_agent(creer_llm("superviseur"), tools=OUTILS_SOUS_AGENTS,
                           system_prompt=PROMPT_SUPERVISEUR, name="superviseur")

AGENT_DE_L_OUTIL = {"deleguer_recherche": "chercheur", "deleguer_redaction": "redacteur", "deleguer_relecture": "relecteur"}


DERNIER_ETAT_SUPERVISEUR = None


def executer_superviseur(question: str) -> dict:
    """Exécute le superviseur ; en cas de dépassement de la limite de récursion, conserve le dernier état."""
    global DERNIER_ETAT_SUPERVISEUR
    reinitialiser_espace(question)
    etat = {"messages": []}
    try:
        for etat in superviseur.stream({"messages": [HumanMessage(question)]}, {"recursion_limit": 40},
                                       stream_mode="values"):
            DERNIER_ETAT_SUPERVISEUR = etat
    except GraphRecursionError:
        print("⚠️ Limite de récursion atteinte : on conserve le dernier brouillon.")
    # Trajectoire reconstruite à partir de l'historique du superviseur
    trajectoire = []
    for m in etat["messages"]:
        if isinstance(m, AIMessage):
            trajectoire.append("superviseur")
        elif isinstance(m, ToolMessage):
            trajectoire.append(AGENT_DE_L_OUTIL.get(m.name, m.name))
    trajectoire.append("FIN")
    for m in etat["messages"]:
        if isinstance(m, ToolMessage) and AFFICHER_DELEGATIONS:
            print(f"   📋 superviseur ──[{m.name}]──▶ {AGENT_DE_L_OUTIL.get(m.name, m.name):10s} | {texte(m)[:90]!r}")
    return {"reponse": ESPACE["brouillon"],
            "trajectoire": trajectoire,
            "nb_passages_main": sum(isinstance(m, ToolMessage) for m in etat["messages"]),
            "nb_messages_historique": len(etat["messages"]),
            "nb_versions_brouillon": ESPACE["versions"]}


AFFICHER_DELEGATIONS = True

In [ ]:
# Visualisation : graphe du superviseur (boucle modèle ↔ outils) et du sous-agent chercheur
from IPython.display import Image, Markdown, display
for nom, g in [("superviseur", superviseur), ("sous-agent chercheur", agent_chercheur)]:
    print(f"— Graphe du {nom} —")
    try:
        display(Image(g.get_graph().draw_mermaid_png(max_retries=3, retry_delay=2.0)))
    except Exception as e:
        print(f"(Rendu PNG indisponible : {type(e).__name__}) — code Mermaid :\n{g.get_graph().draw_mermaid()}")

## 4. Première exécution commentée (requête R1)

In [ ]:
resultat_sup_r1 = mesurer("Superviseur (sous-agents outils)", executer_superviseur, REQUETES[0])
print("\n📝 Réponse finale :")
display(Markdown(resultat_sup_r1["reponse"] or "_(vide)_"))

In [ ]:
# Historique vu par le SUPERVISEUR : il ne contient que ses décisions et des comptes rendus condensés
import pandas as pd
pd.set_option("display.max_colwidth", 90)
lignes = []
for i, m in enumerate((DERNIER_ETAT_SUPERVISEUR or {"messages": []})["messages"]):
    appels = ", ".join(tc["name"] for tc in getattr(m, "tool_calls", []) or [])
    lignes.append({"#": i, "type": m.type, "outil": getattr(m, "name", None) or "", "appels d'outils": appels,
                   "taille (car.)": len(texte(m)), "aperçu": texte(m)[:80].replace("\n", " ")})
display(pd.DataFrame(lignes))

### ✍️ Questions (R1)
1. Comparez la trajectoire avec celle du réseau : laquelle est la plus **prévisible** ? la plus **courte** ?
2. Combien de messages contient l'historique du superviseur ? Que « voit » chaque sous-agent ?
3. Où se trouve le risque de **goulet d'étranglement** dans cette architecture ?

## 5. Campagne de mesure sur les 3 requêtes

In [ ]:
resultats_superviseur = [resultat_sup_r1]
for requete in REQUETES[1:]:
    resultats_superviseur.append(mesurer("Superviseur (sous-agents outils)", executer_superviseur, requete))
sauvegarder_resultats(resultats_superviseur, "resultats_superviseur.json")

In [ ]:
for r in resultats_superviseur:
    display(Markdown(f"### {r['requete_id']} — {r['question']}\n\n**Trajectoire :** {' → '.join(r['trajectoire'])}\n\n"
                     f"**Avis du juge :** {r['evaluation_juge'].get('commentaire', '')}\n\n---\n\n{r['reponse']}"))

## 6. Récupération des résultats de l'architecture en réseau (Notebook 1)

* Si `resultats_reseau.json` (produit par le Notebook 1) est présent **et** a été obtenu avec le **même fournisseur/modèle**,
  il est réutilisé.
* Sinon (autre session Colab, autre modèle…), l'architecture en réseau est **ré-exécutée ici automatiquement** :
  la cellule suivante écrit le module `ia102_reseau.py`, copie conforme du code du Notebook 1.

In [ ]:
%%writefile ia102_reseau.py
"""ia102_reseau.py — Architecture en RÉSEAU AUTONOME (handoffs via Command).
Copie conforme des cellules du Notebook 1 (sections 1 à 5)."""
import operator
from typing import Annotated

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.errors import GraphRecursionError
from langgraph.types import Command

from ia102_commun import OUTILS_RECHERCHE, creer_llm, texte

AFFICHER_TRANSFERTS = True   # affiche chaque passage de main en direct
MAX_TRANSFERTS = 8           # garde-fou : nombre max de passages de main par requête
MAX_TOURS_OUTILS = 6         # nb max d'appels LLM consécutifs d'un agent avant transfert forcé


class EtatReseau(MessagesState):
    notes: str
    brouillon: str
    reponse_finale: str
    trajectoire: Annotated[list[str], operator.add]
    nb_transferts: int


def creer_outil_transfert(cible: str, description: str):
    """Fabrique un outil de transfert vers l'agent `cible`."""
    @tool(f"transferer_vers_{cible}", description=description)
    def transfert(message: str) -> str:
        return f"Transfert vers {cible} effectué."
    return transfert


TRANSFERTS = {
    "chercheur": creer_outil_transfert(
        "chercheur", "Passe la main au CHERCHEUR. `message` : les informations précises à rechercher ou vérifier."),
    "redacteur": creer_outil_transfert(
        "redacteur", "Passe la main au RÉDACTEUR. `message` : notes de recherche structurées et sourcées, "
                     "ou liste des corrections à apporter au brouillon."),
    "relecteur": creer_outil_transfert(
        "relecteur", "Passe la main au RELECTEUR. `message` : le brouillon COMPLET de la réponse à relire."),
}


@tool
def publier_reponse_finale(commentaire: str) -> str:
    """Publie le dernier brouillon du rédacteur comme réponse finale et termine le travail.
    `commentaire` : justification courte de la validation."""
    return "Réponse publiée."


EQUIPE = ("Tu fais partie d'une équipe de trois agents autonomes (chercheur, redacteur, relecteur) qui répond "
          "aux questions de la direction de l'entreprise Heliora à partir de sa base documentaire interne. "
          "Il n'y a pas de chef : chaque agent décide lui-même à qui passer la main. L'historique de la conversation "
          "est partagé par toute l'équipe. Tu termines TOUJOURS ton tour en appelant un outil de transfert.\n\n")

PROMPT_CHERCHEUR = EQUIPE + """Tu es le CHERCHEUR.
Ton rôle : collecter les faits et chiffres nécessaires, chacun avec sa source [DOC-xx].
- Utilise rechercher_documents (plusieurs requêtes courtes et ciblées si besoin) et lire_document pour lire un document en entier.
- Attention aux versions : si un document rectifie un chiffre d'un autre, retiens le chiffre rectifié et signale-le.
- Ignore les documents hors sujet.
- Quand tu as assez d'éléments, appelle transferer_vers_redacteur avec dans `message` des notes structurées (faits + chiffres + [DOC-xx]).
- N'écris jamais la réponse finale toi-même."""

PROMPT_REDACTEUR = EQUIPE + """Tu es le RÉDACTEUR.
Ton rôle : rédiger en français la réponse à la question initiale, à partir des notes du chercheur
et en tenant compte des éventuelles remarques du relecteur.
- Réponse structurée (titres courts, puces), précise et chiffrée ; chaque fait est suivi de sa source [DOC-xx] ;
  termine par une ligne « Sources : … ».
- Quand le brouillon est prêt : appelle transferer_vers_relecteur avec le brouillon INTÉGRAL dans `message`.
- S'il manque des informations indispensables : appelle transferer_vers_chercheur en précisant quoi chercher.
- Tu n'as pas accès à la base documentaire."""

PROMPT_RELECTEUR = EQUIPE + """Tu es le RELECTEUR (contrôle qualité).
Ton rôle : vérifier le dernier brouillon transmis par le rédacteur : exactitude des chiffres par rapport aux
résultats de recherche présents dans l'historique (chiffres rectifiés / à jour), réponse complète à la question,
sources [DOC-xx] citées, clarté.
- Si le brouillon est satisfaisant : appelle publier_reponse_finale.
- S'il doit être corrigé : appelle transferer_vers_redacteur avec la liste précise des corrections.
- S'il manque des faits : appelle transferer_vers_chercheur en précisant ce qu'il faut rechercher.
Sois exigeant mais pragmatique : pas plus de deux cycles de révision."""


TRANSFERT_PAR_DEFAUT = {"chercheur": "transferer_vers_redacteur",
                        "redacteur": "transferer_vers_relecteur",
                        "relecteur": "publier_reponse_finale"}


def _decision_vers_command(nom: str, decision: dict, state: dict, nouveaux: list) -> Command:
    """Traduit la décision de l'agent (appel d'un outil de transfert) en Command LangGraph."""
    args = decision.get("args") or {}
    message = str(args.get("message", args.get("commentaire", "")))
    n = state.get("nb_transferts", 0) + 1
    maj = {"messages": nouveaux, "nb_transferts": n, "trajectoire": [nom]}
    if nom == "chercheur":
        maj["notes"] = message
    if nom == "redacteur" and decision["name"] == "transferer_vers_relecteur":
        maj["brouillon"] = message
    brouillon = maj.get("brouillon", state.get("brouillon", ""))

    if decision["name"] == "publier_reponse_finale":
        cible = END
        maj["reponse_finale"] = brouillon
        maj["trajectoire"] = [nom, "FIN"]
    elif n >= MAX_TRANSFERTS:
        cible = END
        maj["reponse_finale"] = brouillon or message
        maj["trajectoire"] = [nom, "ARRÊT (limite de transferts)"]
    else:
        cible = decision["name"].replace("transferer_vers_", "")
    if AFFICHER_TRANSFERTS:
        print(f"   🔀 {nom:10s} ──[{decision['name']}]──▶ {'FIN' if cible == END else cible}")
    return Command(goto=cible, update=maj)


def creer_agent_reseau(nom: str, prompt: str, outils_metier: list, cibles: list[str], peut_publier: bool = False):
    """Construit la fonction-nœud d'un agent du réseau."""
    outils_transfert = [TRANSFERTS[c] for c in cibles] + ([publier_reponse_finale] if peut_publier else [])
    outils_par_nom = {o.name: o for o in outils_metier}
    noms_transfert = {o.name for o in outils_transfert}
    llm = creer_llm(nom).bind_tools(outils_metier + outils_transfert, tool_choice="any")

    def agent(state: EtatReseau) -> Command:
        nouveaux = []
        for _ in range(MAX_TOURS_OUTILS):
            ai = llm.invoke([SystemMessage(prompt)] + state["messages"] + nouveaux)
            ai.name = nom
            nouveaux.append(ai)
            decision = None
            for tc in ai.tool_calls:
                if tc["name"] in outils_par_nom:                       # outil métier → exécution
                    resultat = outils_par_nom[tc["name"]].invoke(tc["args"])
                    nouveaux.append(ToolMessage(content=str(resultat), tool_call_id=tc["id"], name=tc["name"]))
                elif tc["name"] in noms_transfert and decision is None:  # 1er transfert → décision
                    decision = tc
                    nouveaux.append(ToolMessage(content=f"OK ({tc['name']}).", tool_call_id=tc["id"], name=tc["name"]))
                else:                                                   # outil inconnu / 2e transfert
                    nouveaux.append(ToolMessage(content="Ignoré : un seul transfert par tour, outil non autorisé.",
                                                tool_call_id=tc["id"], name=tc["name"]))
            if decision:
                return _decision_vers_command(nom, decision, state, nouveaux)
            if not ai.tool_calls:                                       # rare avec tool_choice="any"
                nouveaux.append(HumanMessage(f"[{nom}] {texte(ai)[:2000]}"))
                return _decision_vers_command(nom, {"name": TRANSFERT_PAR_DEFAUT[nom], "args": {"message": texte(ai)}}, state, nouveaux)
        # trop d'appels d'outils sans transfert → transfert par défaut
        return _decision_vers_command(nom, {"name": TRANSFERT_PAR_DEFAUT[nom], "args": {"message": "Limite d'itérations atteinte."}}, state, nouveaux)

    agent.__name__ = nom
    return agent


def construire_reseau():
    g = StateGraph(EtatReseau)
    g.add_node("chercheur", creer_agent_reseau("chercheur", PROMPT_CHERCHEUR, OUTILS_RECHERCHE, ["redacteur", "relecteur"]),
               destinations=("redacteur", "relecteur", END))
    g.add_node("redacteur", creer_agent_reseau("redacteur", PROMPT_REDACTEUR, [], ["chercheur", "relecteur"]),
               destinations=("chercheur", "relecteur", END))
    g.add_node("relecteur", creer_agent_reseau("relecteur", PROMPT_RELECTEUR, [], ["chercheur", "redacteur"], peut_publier=True),
               destinations=("chercheur", "redacteur", END))
    g.add_edge(START, "chercheur")
    return g.compile()


graphe_reseau = construire_reseau()


DERNIER_ETAT_RESEAU = None


def executer_reseau(question: str) -> dict:
    """Exécute le réseau ; en cas de dépassement de la limite de récursion, conserve le dernier état."""
    global DERNIER_ETAT_RESEAU
    etat = {"messages": [], "brouillon": "", "reponse_finale": "", "trajectoire": [], "nb_transferts": 0}
    try:
        for etat in graphe_reseau.stream(
                {"messages": [HumanMessage(question)], "notes": "", "brouillon": "", "reponse_finale": "",
                 "trajectoire": [], "nb_transferts": 0},
                config={"recursion_limit": 40}, stream_mode="values"):
            DERNIER_ETAT_RESEAU = etat
    except GraphRecursionError:
        print("⚠️ Limite de récursion atteinte : on conserve le dernier brouillon.")
        etat = dict(etat, trajectoire=etat["trajectoire"] + ["ARRÊT (récursion)"])
    return {"reponse": etat.get("reponse_finale") or etat.get("brouillon", ""),
            "trajectoire": etat["trajectoire"],
            "nb_passages_main": etat["nb_transferts"],
            "nb_messages_historique": len(etat["messages"])}

In [ ]:
import json, pathlib
REEXECUTER_RESEAU = "auto"  # @param ["auto", "toujours", "jamais"]

resultats_reseau = None
chemin = pathlib.Path("resultats_reseau.json")
if REEXECUTER_RESEAU != "toujours" and chemin.exists():
    candidats = json.loads(chemin.read_text(encoding="utf-8"))
    memes = {(r["provider"], r["modele"]) for r in candidats} == {(CONFIG.provider, CONFIG.modele)}
    if memes or REEXECUTER_RESEAU == "jamais":
        resultats_reseau = candidats
        print(f"✅ Résultats du Notebook 1 réutilisés ({len(candidats)} requêtes, modèle {candidats[0]['modele']}).")
    else:
        print("⚠️ resultats_reseau.json a été produit avec un autre fournisseur/modèle → ré-exécution pour une comparaison équitable.")

if resultats_reseau is None:
    print("▶️ Exécution de l'architecture en réseau sur les 3 requêtes (même banc de mesure)…")
    import ia102_reseau
    importlib.reload(ia102_reseau)
    resultats_reseau = [mesurer("Réseau (handoffs)", ia102_reseau.executer_reseau, rq) for rq in REQUETES]
    sauvegarder_resultats(resultats_reseau, "resultats_reseau.json")

## 7. 📊 Tableau comparatif Réseau vs Superviseur

Trois vues :
1. **Tableau détaillé** par requête et par critère (avec l'architecture la plus performante sur chaque ligne) ;
2. **Tableau de synthèse** (totaux et moyennes sur les 3 requêtes) ;
3. **Trajectoires** côte à côte.

Conventions : pour les coûts (appels, tokens, durée, étapes), **moins = mieux** ; pour la qualité, **plus = mieux**.

In [ ]:
ARCHI_R, ARCHI_S = "Réseau (handoffs)", "Superviseur (sous-agents outils)"
CRITERES = [  # (libellé, clé, sens : -1 = moins c'est mieux, +1 = plus c'est mieux, format)
    ("Étapes de la trajectoire", "nb_etapes", -1, "{:.0f}"),
    ("Passages de main / délégations", "nb_passages_main", -1, "{:.0f}"),
    ("Appels LLM", "nb_appels_llm", -1, "{:.0f}"),
    ("Tokens d'entrée", "tokens_entree", -1, "{:,.0f}"),
    ("Tokens de sortie", "tokens_sortie", -1, "{:,.0f}"),
    ("Tokens total", "tokens_total", -1, "{:,.0f}"),
    ("Appels d'outils de recherche", "nb_appels_outils", -1, "{:.0f}"),
    ("Durée (s)", "duree_s", -1, "{:.1f}"),
    ("Couverture des points clés", "couverture_points_cles", +1, "{:.0%}"),
    ("Sources citées (nb)", "nb_sources", +1, "{:.1f}"),
    ("Note du LLM-juge (/5)", "note_juge", +1, "{:.2f}"),
]
for r in resultats_reseau + resultats_superviseur:
    r["nb_sources"] = len(r["sources_citees"])

par_cle = {(r["architecture"], r["requete_id"]): r for r in resultats_reseau + resultats_superviseur}


def _gagnant(vr, vs, sens):
    if vr is None or vs is None or vr == vs:
        return "="
    return ARCHI_R.split()[0] if (vr - vs) * sens > 0 else ARCHI_S.split()[0]


lignes = []
for rq in REQUETES:
    rr, rs = par_cle[(ARCHI_R, rq["id"])], par_cle[(ARCHI_S, rq["id"])]
    for libelle, cle, sens, fmt in CRITERES:
        vr, vs = rr.get(cle), rs.get(cle)
        ecart = f"{(vs - vr) / vr:+.0%}" if isinstance(vr, (int, float)) and isinstance(vs, (int, float)) and vr else "—"
        lignes.append({"Requête": f"{rq['id']} · {rq['type']}", "Critère": libelle,
                       "Réseau": fmt.format(vr) if vr is not None else "—",
                       "Superviseur": fmt.format(vs) if vs is not None else "—",
                       "Écart Sup. vs Rés.": ecart, "Meilleur": _gagnant(vr, vs, sens)})
tableau_detaille = pd.DataFrame(lignes).set_index(["Requête", "Critère"])
display(tableau_detaille)

In [ ]:
# Synthèse sur les 3 requêtes : totaux pour les coûts, moyennes pour la qualité
synthese = []
for libelle, cle, sens, fmt in CRITERES:
    vals = {}
    for archi, res in [(ARCHI_R, resultats_reseau), (ARCHI_S, resultats_superviseur)]:
        xs = [r.get(cle) for r in res if r.get(cle) is not None]
        vals[archi] = (sum(xs) / len(xs) if sens > 0 else sum(xs)) if xs else None
    vr, vs = vals[ARCHI_R], vals[ARCHI_S]
    synthese.append({"Critère": libelle + (" — moyenne" if sens > 0 else " — total"),
                     "Réseau": fmt.format(vr) if vr is not None else "—",
                     "Superviseur": fmt.format(vs) if vs is not None else "—",
                     "Écart Sup. vs Rés.": f"{(vs - vr) / vr:+.0%}" if vr and vs is not None else "—",
                     "Meilleur": _gagnant(vr, vs, sens)})
tableau_synthese = pd.DataFrame(synthese).set_index("Critère")
display(tableau_synthese)

trajectoires = pd.DataFrame([{
    "Requête": rq["id"],
    "Réseau (handoffs)": " → ".join(par_cle[(ARCHI_R, rq["id"])]["trajectoire"]),
    "Superviseur": " → ".join(par_cle[(ARCHI_S, rq["id"])]["trajectoire"]),
} for rq in REQUETES]).set_index("Requête")
display(trajectoires)

### Répartition des appels LLM et des tokens par agent
Qui consomme ? Dans le réseau, chaque agent relit l'historique commun ; dans l'architecture supervisée,
le superviseur est sollicité entre chaque délégation (goulet potentiel) mais les spécialistes travaillent sur un contexte réduit.

In [ ]:
rep = []
for r in resultats_reseau + resultats_superviseur:
    for agent, n in r["appels_par_agent"].items():
        rep.append({"Architecture": r["architecture"].split()[0], "Requête": r["requete_id"], "Agent": agent,
                    "Appels LLM": n, "Tokens": r["tokens_par_agent"].get(agent, 0)})
repartition = (pd.DataFrame(rep).pivot_table(index="Agent", columns="Architecture",
                                             values=["Appels LLM", "Tokens"], aggfunc="sum", fill_value=0))
display(repartition)

In [ ]:
# Graphiques : une métrique par panneau (pas de double axe), une couleur fixe par architecture
import matplotlib.pyplot as plt
import numpy as np

COULEURS = {ARCHI_R: "#2a78d6", ARCHI_S: "#eb6834"}
PANNEAUX = [("nb_appels_llm", "Appels LLM"), ("tokens_total", "Tokens total"),
            ("couverture_points_cles", "Couverture des points clés"), ("note_juge", "Note du LLM-juge (/5)")]
x = np.arange(len(REQUETES)); larg = 0.38
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
for ax, (cle, titre) in zip(axes.flat, PANNEAUX):
    for j, (archi, res) in enumerate([(ARCHI_R, resultats_reseau), (ARCHI_S, resultats_superviseur)]):
        vals = [(r.get(cle) or 0) for r in res]
        barres = ax.bar(x + (j - 0.5) * larg, vals, larg * 0.94, color=COULEURS[archi], label=archi)
        etiquettes = [f"{v:.0%}" if cle == "couverture_points_cles" else f"{v:.1f}" if cle == "note_juge" else f"{v:,.0f}" for v in vals]
        ax.bar_label(barres, labels=etiquettes, fontsize=8, color="#52514e", padding=2)
    ax.set_title(titre, fontsize=11, loc="left", color="#0b0b0b")
    ax.set_xticks(x, [rq["id"] for rq in REQUETES])
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.8); ax.set_axisbelow(True)
    if cle == "couverture_points_cles":
        ax.set_ylim(0, 1.15)
        ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    if cle == "note_juge":
        ax.set_ylim(0, 5.5)
poignees, libelles = axes[0, 0].get_legend_handles_labels()
fig.legend(poignees, libelles, loc="upper right", ncol=2, frameon=False, fontsize=9)
fig.suptitle(f"Réseau vs Superviseur — {CONFIG.provider} / {CONFIG.modele}", fontsize=12, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig("comparaison_architectures.png", dpi=150)
plt.show()

## 8. Export du livrable « tableau comparatif »

Fichiers produits (panneau 📁 de Colab) :
* `tableau_comparatif.xlsx` — onglets *Synthèse*, *Détail par requête*, *Trajectoires*, *Répartition par agent*, *Réponses* ;
* `tableau_comparatif.md` — version Markdown (à coller dans un compte rendu) ; `tableau_comparatif.csv` ;
* `comparaison_architectures.png` — graphiques ; `resultats_reseau.json`, `resultats_superviseur.json` — données brutes.

In [ ]:
reponses = pd.DataFrame([{"Architecture": r["architecture"], "Requête": r["requete_id"], "Question": r["question"],
                          "Réponse finale": r["reponse"], "Points manquants": "; ".join(r["points_manquants"]),
                          "Avis du juge": r["evaluation_juge"].get("commentaire", "")}
                         for r in resultats_reseau + resultats_superviseur])
with pd.ExcelWriter("tableau_comparatif.xlsx") as xl:
    tableau_synthese.to_excel(xl, sheet_name="Synthèse")
    tableau_detaille.to_excel(xl, sheet_name="Détail par requête")
    trajectoires.to_excel(xl, sheet_name="Trajectoires")
    repartition.to_excel(xl, sheet_name="Répartition par agent")
    reponses.to_excel(xl, sheet_name="Réponses", index=False)
tableau_detaille.reset_index().to_csv("tableau_comparatif.csv", index=False, encoding="utf-8-sig")

entete = (f"# Tableau comparatif — Réseau (handoffs) vs Superviseur (sous-agents outils)\n\n"
          f"Fournisseur : `{CONFIG.provider}` — modèle : `{CONFIG.modele}` — 3 requêtes de test\n\n")
md_txt = (entete + "## Synthèse (3 requêtes)\n\n" + tableau_synthese.to_markdown() +
          "\n\n## Trajectoires\n\n" + trajectoires.to_markdown() +
          "\n\n## Détail par requête\n\n" + tableau_detaille.to_markdown() + "\n")
pathlib.Path("tableau_comparatif.md").write_text(md_txt, encoding="utf-8")
print("💾 tableau_comparatif.xlsx, tableau_comparatif.md, tableau_comparatif.csv, comparaison_architectures.png écrits")

In [ ]:
# @title Téléchargement du livrable (optionnel)
TELECHARGER = False  # @param {type:"boolean"}
if TELECHARGER:
    try:
        from google.colab import files
        for f in ["tableau_comparatif.xlsx", "tableau_comparatif.md", "comparaison_architectures.png"]:
            files.download(f)
    except ImportError:
        print("Hors Colab : les fichiers sont dans le répertoire courant.")

## 9. Lecture guidée des résultats

La cellule ci-dessous calcule automatiquement quelques constats à partir de **vos** mesures
(elles varient selon le modèle et d'une exécution à l'autre : les LLM ne sont pas déterministes).

In [ ]:
def total(res, cle):
    return sum((r.get(cle) or 0) for r in res)

def moyenne(res, cle):
    xs = [r.get(cle) for r in res if r.get(cle) is not None]
    return sum(xs) / len(xs) if xs else float("nan")

tr, ts = total(resultats_reseau, "tokens_total"), total(resultats_superviseur, "tokens_total")
ar, as_ = total(resultats_reseau, "nb_appels_llm"), total(resultats_superviseur, "nb_appels_llm")
constats = [
    f"- **Tokens** : le superviseur consomme {ts:,} tokens contre {tr:,} pour le réseau ({(ts - tr) / max(tr, 1):+.0%}).",
    f"- **Appels LLM** : {as_} (superviseur) contre {ar} (réseau) ({(as_ - ar) / max(ar, 1):+.0%}).",
    f"- **Tokens moyens par appel** : {ts / max(as_, 1):,.0f} (superviseur) contre {tr / max(ar, 1):,.0f} (réseau) "
    f"→ effet de l'historique partagé vs contextes isolés.",
    f"- **Qualité** : couverture moyenne {moyenne(resultats_superviseur, 'couverture_points_cles'):.0%} (superviseur) "
    f"vs {moyenne(resultats_reseau, 'couverture_points_cles'):.0%} (réseau) ; note juge "
    f"{moyenne(resultats_superviseur, 'note_juge'):.2f} vs {moyenne(resultats_reseau, 'note_juge'):.2f}.",
]
sup_calls = sum(r["appels_par_agent"].get("superviseur", 0) for r in resultats_superviseur)
constats.append(f"- **Goulet d'étranglement** : le superviseur représente {sup_calls / max(as_, 1):.0%} des appels LLM de son architecture.")
if CONFIG.provider == "simulation":
    constats.append("- ⚠️ *Mode simulation : ces chiffres illustrent la mécanique de mesure, pas la performance réelle des architectures.*")
display(Markdown("\n".join(constats)))

### 🧾 Grille d'analyse à compléter (livrable)

| Critère | Réseau autonome (handoffs) | Superviseur (sous-agents outils) |
|---|---|---|
| Prévisibilité / explicabilité de la trajectoire | | |
| Coût (appels LLM, tokens) | | |
| Qualité des réponses (R1 factuelle, R2 synthèse, R3 décision) | | |
| Robustesse (boucles, erreurs, arrêt) | | |
| Goulet d'étranglement / point unique de défaillance | | |
| Facilité d'ajout d'un nouvel agent | | |
| Cas d'usage le plus adapté | | |

### 🔭 Extension : vers une hiérarchie à plusieurs niveaux
Une **équipe** entière peut devenir un outil d'un superviseur de niveau supérieur. Exemple (à tester) :
```python
equipe_redaction = create_agent(creer_llm("chef_redaction"),
                                tools=[deleguer_redaction, deleguer_relecture],
                                system_prompt="Tu diriges l'équipe rédaction : fais rédiger puis relire jusqu'à validation.")

@tool
def deleguer_a_l_equipe_redaction(consignes: str) -> str:
    """Confie la production d'une réponse relue et validée à l'équipe rédaction."""
    res = equipe_redaction.invoke({"messages": [HumanMessage(consignes)]}, {"recursion_limit": 25})
    return texte(res["messages"][-1])

directeur = create_agent(creer_llm("directeur"), tools=[deleguer_recherche, deleguer_a_l_equipe_redaction],
                         system_prompt="Tu es le directeur : fais rechercher, puis confie la rédaction à l'équipe rédaction.")
```
Mesurez ensuite le surcoût (appels du « chef d'équipe ») et l'effet sur la qualité.

### 🧠 À retenir
* **Superviseur** = orchestration centralisée : trajectoire plus **prévisible** et **traçable**, contextes **isolés**
  (chaque spécialiste ne voit que sa consigne), arrêt contrôlé.
* **Limites** : le superviseur est **sollicité entre chaque étape** (appels supplémentaires, latence) et constitue
  un **goulet d'étranglement** et un **point unique de défaillance** ; risque de perte d'information lors des délégations.
* **Réseau** : plus flexible pour les problèmes non séquentiels, mais moins prévisible et souvent plus coûteux en
  tokens (historique partagé relu par tous).
* Alternatives : **hiérarchies à plusieurs niveaux** (équipes de superviseurs), **swarm** (handoffs + mémoire de l'agent actif),
  **workflows déterministes** quand le processus est connu à l'avance.